In [6]:
import os
import numpy as np
import tensorflow as tf
import pickle

BASE = r'C:\Users\c.villalobos-ortiz\Desktop\AIModel'
os.chdir(BASE)
print(f'📁 Directorio actual: {os.getcwd()}')

archivos = [
    'models/v1/modelo_lsm.keras',
    'models/v1/encoder_lsm.pkl'
]

for archivo in archivos:
    if os.path.exists(archivo):
        tamaño = os.path.getsize(archivo) / 1024
        print(f'✅ {archivo} ({tamaño:.1f} KB)')
    else:
        print(f'❌ No encontrado: {archivo}')

📁 Directorio actual: C:\Users\c.villalobos-ortiz\Desktop\AIModel
✅ models/v1/modelo_lsm.keras (750.0 KB)
✅ models/v1/encoder_lsm.pkl (0.3 KB)


In [7]:
modelo = tf.keras.models.load_model('models/v1/modelo_lsm.keras')
print('✅ Modelo cargado')

converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Optimización para dispositivos pequeños
tflite_model = converter.convert()

os.makedirs('models/v1', exist_ok=True)
with open('models/v1/modelo_lsm.tflite', 'wb') as f:
    f.write(tflite_model)

tamaño_keras = os.path.getsize('models/v1/modelo_lsm.keras') / 1024
tamaño_tflite = os.path.getsize('models/v1/modelo_lsm.tflite') / 1024
print(f'✅ Modelo convertido')
print(f'📁 .keras:  {tamaño_keras:.1f} KB')
print(f'📁 .tflite: {tamaño_tflite:.1f} KB')
print(f'📉 Reducción: {(1 - tamaño_tflite/tamaño_keras)*100:.1f}%')

✅ Modelo cargado
INFO:tensorflow:Assets written to: C:\Users\C1E0C~1.VIL\AppData\Local\Temp\tmp8j906q9b\assets


INFO:tensorflow:Assets written to: C:\Users\C1E0C~1.VIL\AppData\Local\Temp\tmp8j906q9b\assets


Saved artifact at 'C:\Users\C1E0C~1.VIL\AppData\Local\Temp\tmp8j906q9b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 63), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 21), dtype=tf.float32, name=None)
Captures:
  2376054384496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054389424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054745472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054746880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054740896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054743184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054746000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054841488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054844304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2376054846064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  23760548

In [8]:
with open('models/v1/encoder_lsm.pkl', 'rb') as f:
    encoder = pickle.load(f)

# Cargar TFLite
interpreter = tf.lite.Interpreter(model_path='models/v1/modelo_lsm.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test con dato de prueba
test_input = np.random.rand(1, 63).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
letra = encoder.classes_[output.argmax()]

print(f'✅ TFLite funciona correctamente')
print(f'   Input shape: {input_details[0]["shape"]}')
print(f'   Output shape: {output_details[0]["shape"]}')
print(f'   Test predicción: {letra}')

✅ TFLite funciona correctamente
   Input shape: [ 1 63]
   Output shape: [ 1 21]
   Test predicción: I


C:\Users\c.villalobos-ortiz\.conda\envs\lsm\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
